<h2 align="center"><b>Simulating the N-Body problem</b></h2>
<div align=center>
<table style="width: 100%; border: none; border-collapse: collapse;">
<tr>

<style>
.emines-logo-wrap {
    position: relative;
    display: inline-block;
}

.emines-logo-wrap .emines-logo-sizer {
    visibility: hidden;
    display: block;
    width: auto;
}

.emines-logo-wrap .emines-logo-fill {
    position: absolute;
    inset: 0;
    background-color: #000000; /* fallback: light theme */
    -webkit-mask-image: url('../workshop_utils/assets/emines_logo.svg');
    mask-image: url('../workshop_utils/assets/emines_logo.svg');
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-size: contain;
    mask-size: contain;
    -webkit-mask-position: center;
    mask-position: center;
}
@media (prefers-color-scheme: dark) {
    .emines-logo-wrap .emines-logo-fill { background-color: #ffffff; }
}
.emines-logo-wrap .emines-logo-fill {
    background-color: var(--vscode-foreground, var(--logo-color));
}
</style>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<!-- Left side -->
<td style="width: 15%; border: none; text-align: right; vertical-align: middle;">
<div style="padding-right: 0;" class="emines-logo-wrap">
    <img class="emines-logo-sizer" src="../workshop_utils/assets/emines_logo.svg" alt="" aria-hidden="true">
    <div class="emines-logo-fill" role="img" aria-label="EMINES logo"></div>
</div>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<!-- Video -->
<td style="width: 40%; border: none; text-align: center; vertical-align: middle;">
  <video
    src="../workshop_utils/assets/nbody_demo.mp4"
    autoplay
    muted
    loop
    playsinline
    width="600">
  </video>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

<style>
.ephysics-logo-wrap {
    position: relative;
    display: inline-block;
}

.ephysics-logo-wrap .ephysics-logo-sizer {
    visibility: hidden;
    display: block;
    width: auto;
}

.ephysics-logo-wrap .ephysics-logo-fill {
    position: absolute;
    inset: 0;
    background-color: #000000; /* fallback: light theme */
    -webkit-mask-image: url('../workshop_utils/assets/ephysics_logo.png');
    mask-image: url('../workshop_utils/assets/ephysics_logo.png');
    -webkit-mask-repeat: no-repeat;
    mask-repeat: no-repeat;
    -webkit-mask-size: contain;
    mask-size: contain;
    -webkit-mask-position: center;
    mask-position: center;
}

@media (prefers-color-scheme: dark) {
    .ephysics-logo-wrap .ephysics-logo-fill { background-color: #ffffff; }
}

.ephysics-logo-wrap .ephysics-logo-fill {
    background-color: var(--vscode-foreground, var(--logo-color));
}
</style>

<!-- Right side -->
<td style="width: 15%; border: none; text-align: left; vertical-align: middle;">
<div style="padding-left: 0;" class="ephysics-logo-wrap">
    <img class="ephysics-logo-sizer" src="../workshop_utils/assets/ephysics_logo.png" alt="" aria-hidden="true">
    <div class="ephysics-logo-fill" role="img" aria-label="EPhy logo"></div>
</div>
</td>

<td style="width: 7.5%; border: none; text-align: center; vertical-align: middle;"></td>

  </tr>
</table>
</div>

In this notebook, We will simulate a system involving several bodies attracting one another through gravity.\
To calculate the force exerted by a body $A$ on a body $B$, we will use the newtonian gravtitational force formula given by:
$$\vec{F}_{A/B}=\frac{\mathrm{G} \ m_A \ m_B}{||\vec{r}_A-\vec{r}_B||^3} \ (\vec{r}_A-\vec{r}_B)$$

We will use <b>numpy</b> for vector representations and operations.<br>
We will use <b>pygame</b> to visualize the bodies in real time.

### The simulation recipe

- **State:** every body has a position, velocity, and mass.
- **Model:** each body accelerates toward every other body.
- **Method:** Euler is simple but drifts; velocity-Verlet uses the same state but keeps energy much better.
- **Visualization:** pygame draws the bodies and their trails in real time.

In [ ]:
from collections import deque
import sys

import numpy as np
import pygame

import matplotlib
from workshop_utils.find_backend import use_window_backend
if "matplotlib.pyplot" not in sys.modules: use_window_backend(matplotlib)
import matplotlib.pyplot as plt

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Defining the system — Determining state variables — Choosing parameters</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

The subject system is a set of $\bm{N}$ bodies with masses $\bm{m_0}, \bm{m_2}, ..., \bm{m_{N-1}}$\
We are studying the evolution of the position vector $\bm{\vec{r}_i}$ over **time** relative to the body $\bm{i}$.\
The state here is similar to that of the bouncing ball system. We just have to include all the bodies.\
The state can be represented by $ \mathcal{S}=\{\ (\bm{\vec{r}_i}, \ \bm{\frac{d\vec{r}_i}{dt}}), \quad i \in [\![0, N-1]\!] \ \} $.

**System constants/parameters:**
- `G` is the newtonian gravitational constant.
- `N` is the number of bodies.
- The masses $ \{\ \bm{m_i}, \ i \in [\![0, N-1]\!] \ \} $ are represented by the numpy array `masses`.

**Simulation parameters:**
- `dt` is the timestep in seconds.
- `integrator` is the type of integrator, leave as `"euler"` for now.

**Visualization parameters:**
- `WINDOW_SIZE` of the pygame instance.
- `fps` is the number of frames to render and display per second.
- `timescale` represents how much will the simulation be **slowed down** comparing to real life.
- `radiuses` is an array representing the visual radiuses of the bodies that will be draw as circles.\
(Note that this has nothing to do with the physical body, as they are treated as mass points)
- `colors` is the array of `(R, G, B)` colors of the bodies
- `TRAIL_LENGTH` of the bodies.
- `WORLD_WIDTH` helps mapping real body coordinates to pixels on the simulation window.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
You can modify system and simulation parameters below
</span></div>
<br>

***

In [ ]:
G = 1.0
N = 3
masses = np.array([1, 1, 1])

dt = 0.001
integrator = "euler"  # try "verlet" after observing Euler

WINDOW_SIZE = (900, 700)
fps = 30
timescale = 1
radiuses = [9, 9, 9]
colors = [
    (255, 95, 95),
    (110, 220, 255),
    (255, 220, 100)
    ]
TRAIL_LENGTH = 500
WORLD_WIDTH = 3.2  # width of the visible world

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Initial conditions - Boundary conditions</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

Similarily to the previous example, the initial conditions are the initial position and velocity, but for each of the bodies.

The initial condition is therefore: $ \{\ (\ \bm{\vec{r}_i}(0), \ \bm{\frac{d\vec{r}_i}{dt}}(0) \ ), \quad i \in [\![0, N-1]\!] \ \} $.

In code, we can represent them using `initial_positions` and `initial_velocities` arrays of shape `(N, 2)`.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you can choose an initial condition or make your own.
</span></div>
<br>

***

In [ ]:
# --- Figure-eight (periodic) --- #

initial_positions = np.array([
    [-0.97000436,  0.24308753],
    [ 0.0,         0.0       ],
    [ 0.97000436, -0.24308753],
])

initial_velocities = np.array([
    [ 0.466203685, 0.43236573],
    [-0.93240737, -0.86473146],
    [ 0.466203685, 0.43236573],
])

# --- Equilateral triangle (periodic) --- #

# radius = 0.8
# speed = np.sqrt(G * masses[0] / (np.sqrt(3) * radius))
# initial_positions = np.array([
#     [ radius,                0.0],
#     [-radius / 2,  np.sqrt(3) * radius / 2],
#     [-radius / 2, -np.sqrt(3) * radius / 2],
# ])
# initial_velocities = np.array([
#     [          0.0,  speed],
#     [-np.sqrt(3) * speed / 2, -speed / 2],
#     [ np.sqrt(3) * speed / 2, -speed / 2],
# ])

# --- Broucke A7 (periodic) --- #

# initial_positions = np.array([
#     [-0.109549476941, -0.000203250735],
#     [ 1.661316429463,  0.000032207895],
#     [-1.551766952522,  0.000171042950],
# ])

# initial_velocities = np.array([
#     [ 0.000033179640,  0.991346842049],
#     [ 0.000085239069, -0.156997716798],
#     [-0.000118418708, -0.834349125251],
# ])

# --- Make your own --- # (optional)

...

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Acceleration - Energy</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

As the bodies change position and relative distance, it is important to keep updating their accelerations after each time step.

For body *i*, the acceleration is:
$$\mathbf{\vec{a_i}} = \sum_{j\ne i} G \ m_j\,\frac{\vec{r}_j-\vec{r}_i}{||\vec{r}_j-\vec{r}_i||^3}$$

A method we can use to judge the stability of our system is measuring the relative energy drift.\
Since our system is physically conservative, it should theoretically preserve its initial energy.\
Buy measuring tracking the simulated system energy, we can have an idea about the efficiency of our method.

For body *i*, the energy is:
$$\mathbf{E}_i = \frac{1}{2} \ m_i \ v_i^2 - \sum_{j\ne i} \frac{1}{2} \ G \ m_i \ m_j\,\frac{1}{||\vec{r}_j-\vec{r}_i||}$$

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will implement <b>acceleration</b> and <b>energy</b> formulae.
</span></div>
<br>

***

In [ ]:
def get_accelerations(positions, masses, G):
    N = len(masses)
    accelerations = np.zeros((N, 2))

    for i in range(N):
        for j in range(N):
            if i == j:
                continue

            delta_x = ...
            delta_y = ...

            distance = ...
            factor = ...

            accelerations[i, 0] += ...
            accelerations[i, 1] += ...

    return accelerations


def total_energy(positions, velocities, masses, G):
    N = len(masses)

    kinetic = 0.0
    for i in range(N):
        vx = ...
        vy = ...

        kinetic += ...

    potential = 0.0
    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            delta_x = ...
            delta_y = ...

            distance = ...

            potential -= ...

    return kinetic + potential

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
    <span style="font-size: 24px"><b>Time integration</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid ; margin: 0 10px;">
</div>

We can now calculate the acceleration of every body from the gravitational forces. \
The next step is to **advance the system in time**, updating positions and velocities at each time step **dt**.

**Euler's method** uses the current acceleration to update velocity, and advances position with the old velocity:

$$
\begin{cases}
\vec{v}_{n+1} = \vec{v}_n + \vec{a}_n\,dt \\[0.5em]
\vec{r}_{n+1} = \vec{r}_n + \vec{v}_n\,dt
\end{cases}
$$

A more accurate alternative is **velocity-Verlet**. It first advances the positions using the current acceleration, recalculates the acceleration at the new positions, then updates velocity using the average of both accelerations:

$$
\begin{cases}
\vec{r}_{n+1} = \vec{r}_n + \vec{v}_n\,dt
+ \frac{1}{2}\vec{a}_n\,dt^2 \\[0.8em]
\vec{a}_{n+1} = \vec{a}(\vec{r}_{n+1}) \\[0.8em]
\vec{v}_{n+1} = \vec{v}_n
+ \frac{1}{2}(\vec{a}_n+\vec{a}_{n+1})\,dt
\end{cases}
$$

Velocity-Verlet requires an additional acceleration evaluation per step, but generally provides better long-term energy conservation for gravitational systems.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will implement <b>Euler and velocity-Verlet</b> to evolve the system.
</span></div>
<br>

***

In [ ]:
def euler_step(positions, velocities, masses, G, dt): ##
    accelerations = get_accelerations(positions, masses, G)

    for i in range(N):
        positions[i, 0] += ...
        positions[i, 1] += ...
        velocities[i, 0] += ...
        velocities[i, 1] += ...


def verlet_step(positions, velocities, masses, G, dt):
    old_accelerations = get_accelerations(positions, masses, G)

    for i in range(N):
        positions[i, 0] += ...
        positions[i, 1] += ...

    new_accelerations = get_accelerations(positions, masses, G)

    for i in range(N):
        velocities[i, 0] += ...
        velocities[i, 1] += ...


def advance(positions, velocities, masses, G, dt, method):
    if method == "euler":
        euler_step(positions, velocities, masses, G, dt)
    elif method == "verlet":
        verlet_step(positions, velocities, masses, G, dt)
    else:
        raise ValueError("method must be 'euler' or 'verlet'")


<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>World-to-screen conversion</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

Our simulation calculates the position of each body in world coordinates, but the screen represents positions in pixels. We need to convert between these two coordinate systems to visualize the motion.

We center the world at the middle of the window and use a uniform scale to preserve proportions. Since screen coordinates increase downward, the vertical axis must also be inverted.

For a world position $\vec{r}=(x,y)$, the corresponding screen position is:

$$
\begin{cases}
x_{\text{screen}} = \dfrac{W}{2} + sx
\\[1em]
y_{\text{screen}} = \dfrac{H}{2} - sy
\end{cases}
\qquad
s = \dfrac{\min(W,H)}{L}
$$

where (W) and (H) are the window dimensions, (L) is the world width, and (s) is the scale in pixels per world unit.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will implement the <b>world-to-screen</b> conversion.
</span></div>
<br>

***

In [ ]:
def world_to_screen(position, window_size, world_width):
    width, height = window_size
    scale = ...
    return (
        ...,
        ...
    )

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>Simulation loop</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

We now have everything needed to bring our N-body simulation to life!

The simulation runs inside a continuous loop that combines **numerical integration**, **visualization**, and **energy monitoring**.

At each frame, we:

- Advance the system: Update positions and velocities using the selected integration method. Multiple time steps (substeps) can be performed per frame for smoother and more accurate simulations.

- Render the bodies: Convert world coordinates into screen coordinates, then draw each body and its trajectory.

- Monitor energy conservation: Compare the current total energy with its initial value to evaluate numerical stability.

The relative energy drift is defined as:
$
\qquad
\delta E(t) = \frac{E(t)-E(0)}{|E(0)|}
$

For an ideal conservative system: $\delta E(t)=0$. \
In our numerical simulation, deviations from zero reveal how well the integration method preserves energy.

<br>
<div align="center"><span style="border: 1px solid; padding: 10px; display: inline-block;">
Below, you will complete the <b>simulation loop</b>, <b>visualize</b> the motion, and <b>track energy</b> conservation.
</span></div>
<br>

***

In [ ]:
def run_simulation(initial_positions, initial_velocities, masses, colors, radiuses,
                   method, dt, substeps, G,
                   window_size=WINDOW_SIZE, world_width=WORLD_WIDTH):
    pygame.init()
    screen = pygame.display.set_mode(window_size)
    pygame.display.set_caption("E-Physics Workshop — N-body simulation")
    clock = pygame.time.Clock()
    font = pygame.font.Font(None, 25)

    positions = initial_positions.copy()
    velocities = initial_velocities.copy()
    trails = [deque(maxlen=TRAIL_LENGTH) for _ in range(len(masses))]
    initial_energy = total_energy(positions, velocities, masses, G)
    times, energies = [0.0], [initial_energy]
    simulated_time = 0.0
    running = True

    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        for _ in range(substeps):
            ... # advance the system
            simulated_time += dt
            for trail, position in zip(trails, positions):
                trail.append(position.copy())
        times.append(simulated_time)
        energies.append(total_energy(positions, velocities, masses, G))

        screen.fill((13, 17, 28))
        width, height = window_size
        pygame.draw.line(...) # horizontal line with color (45, 52, 70)
        pygame.draw.line(...) # verical line with color (45, 52, 70)

        for trail, color in zip(trails, colors):
            if len(trail) > 1:
                points = [world_to_screen(point, window_size, world_width) for point in trail]
                pygame.draw.lines(screen, color, False, points, 2)

        for index, position in enumerate(positions): ##
            center = ... # center of the current body in window coordinates
            pygame.draw.circle(...)

        relative_drift = ...
        status = f"{method} | t = {simulated_time:7.2f} | Relative drift = {relative_drift:+.2e}"
        screen.blit(font.render(status, True, (235, 235, 235)), (18, 16))

        pygame.display.flip()
        clock.tick(fps)

    pygame.quit()
    return np.asarray(times), np.asarray(energies), positions, velocities


substeps_per_frame = max(1, round(1 / (fps * timescale * dt)))
time_history, energy_history, final_positions, final_velocities = run_simulation(
    initial_positions, initial_velocities, masses, colors, radiuses, integrator,
    dt, substeps_per_frame, G
)

<div style="display: flex; align-items: center; text-align: center;">
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
    <span style="font-size: 24px"><b>Optional: Energy check</b></span>
    <hr style="flex: 1; border: none; border-top: 2px solid; margin: 0 10px;">
</div>

The plot opens in a separate Matplotlib window.\
A nearly flat line means that the method is preserving total energy.\
Systematic drift means numerical error.
***

In [ ]:
relative_energy_error = (energy_history - energy_history[0]) / abs(energy_history[0])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(time_history, relative_energy_error, color="tab:orange")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_xlabel("simulation time")
ax.set_ylabel("(E − E₀) / |E₀|")
ax.set_title(f"Energy drift with {integrator}")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()